# Making It Production-Ready

In the previous notebook we built a 3-agent DevOps pipeline. It works — but it has problems you'd hit immediately in production. This notebook fixes them, one at a time.

| Section | The Problem | The Fix | CrewAI API |
|---------|------------|---------|------------|
| 1 | Agent output is raw text — parsing it is fragile | **Structured Output** | `output_pydantic=Model` |
| 2 | Bad or vague output passes through unchecked | **Code Guardrail** | `guardrail=validate_fn` |
| 3 | Writing validation functions for everything is tedious | **No-Code Guardrail** | `guardrail="plain English"` |
| 4 | Every run starts from scratch — no learning | **Memory** | `memory=True` |

In [1]:
import os
from typing import Any, Tuple

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai.tasks.task_output import TaskOutput
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

llm = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

LOG_INPUT = """
[2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment
[2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state
[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
[2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints
[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated
"""

log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=True,
)

---

## 1. Structured Output

### The Problem: Raw Text Is Fragile

In Notebook 1, our agent returns a wall of markdown text. Let's run it and try to extract specific fields.

In [2]:
v1_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

v1_crew = Crew(
    agents=[log_analyzer],
    tasks=[v1_task],
    process=Process.sequential,
    verbose=False,
)

v1_result = v1_crew.kickoff(inputs={"log_data": LOG_INPUT})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Analysis Report of Deployment Log**                                                                          │
│                                                                                                                 │
│  **Primary Issue Description:**                                                                                 │
│  The deployment of the application `myapp` failed due to an inability to pull the specified Docker image,       │
│  resulting in a series of errors culminating in the rollback of the deployment.                                 │
│                                                                                                                 │
│  **Key Error Messages and Codes:**                                                                              │
│  1. ERROR: Pod `myapp-deployment-7b8c9d5f4-abc12` failed to start                                               │
│  2. ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require    │
│  'docker login'                                                                                                 │
│  3. ERROR: Pod `myapp-deployment-7b8c9d5f4-abc12` status: ImagePullBackOff                                      │
│  4. ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline              │
│  5. CRITICAL: Production deployment failed - rollback initiated                                                 │
│                                                                                                                 │
│  **Timeline of Failure Events:**                                                                                │
│  - **14:32:15**: Deployment of `myapp-deployment` initiated.                                                    │
│  - **14:32:16**: Pod `myapp-deployment-7b8c9d5f4-abc12` enters Pending state.                                   │
│  - **14:32:17**: Pod `myapp-deployment-7b8c9d5f4-abc12` failed to start.                                        │
│  - **14:32:18**: Image pull failure detected for "myapp:v1.2.3" due to "pull access denied".                    │
│  - **14:32:18**: Pod `myapp-deployment-7b8c9d5f4-abc12` status updated to ImagePullBackOff.                     │
│  - **14:32:25**: Deployment rollout failed; exceeded its progress deadline.                                     │
│  - **14:32:26**: Service `myapp-service` reported no available endpoints.                                       │
│  - **14:32:29**: Critical failure in production deployment resulted in rollback.                                │
│                                                                                                                 │
│  **Root Cause Analysis:**                                                                                       │
│  The primary issue causing the deployment failure was the inability to pull the Docker image "myapp:v1.2.3".    │
│  This was due to either access denial or the non-existence of the repository, as suggested by the error         │
│  message indicating a possible need for `docker login`. This prevented the pod from starting, leading to the    │
│  rollout failure of the deployment.                                                                             │
│                                                        

In [3]:
print(v1_result.raw)

**Analysis Report of Deployment Log**

**Primary Issue Description:**
The deployment of the application `myapp` failed due to an inability to pull the specified Docker image, resulting in a series of errors culminating in the rollback of the deployment. 

**Key Error Messages and Codes:**
1. ERROR: Pod `myapp-deployment-7b8c9d5f4-abc12` failed to start
2. ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
3. ERROR: Pod `myapp-deployment-7b8c9d5f4-abc12` status: ImagePullBackOff
4. ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
5. CRITICAL: Production deployment failed - rollback initiated

**Timeline of Failure Events:**
- **14:32:15**: Deployment of `myapp-deployment` initiated.
- **14:32:16**: Pod `myapp-deployment-7b8c9d5f4-abc12` enters Pending state.
- **14:32:17**: Pod `myapp-deployment-7b8c9d5f4-abc12` failed to start.
- **14:32:18**: Image pull failure detected f

In [4]:
# Want the root cause? Parse the string:
lines = v1_result.raw.split("\n")
for line in lines:
    if "root cause" in line.lower():
        print(f"Found: {line}")
        break

# Want the number of errors? Count manually in free-form text?
# Want to pass typed data to the next agent? Impossible.
# What if the agent formats it differently next run? Everything breaks.

Found: **Root Cause Analysis:**


### The Fix: `output_pydantic`

Define a Pydantic model describing the shape of the output you want. Add `output_pydantic=Model` to the task. CrewAI forces the agent to return data matching that schema.

In [5]:
class ErrorEntry(BaseModel):
    message: str = Field(description="The error message text")
    severity: str = Field(description="ERROR, CRITICAL, or WARNING")
    timestamp: str = Field(description="When the error occurred")


class LogAnalysisReport(BaseModel):
    primary_issue: str = Field(description="One-line description of the main issue")
    root_cause: str = Field(description="Root cause analysis based on log evidence")
    errors: list[ErrorEntry] = Field(description="All errors found in the log")
    affected_components: list[str] = Field(description="System components affected")
    timeline: list[str] = Field(description="Sequence of events leading to failure")

In [6]:
structured_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    agent=log_analyzer,
)

structured_crew = Crew(
    agents=[log_analyzer],
    tasks=[structured_task],
    process=Process.sequential,
    verbose=False,
)

structured_result = structured_crew.kickoff(inputs={"log_data": LOG_INPUT})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "primary_issue": "Deployment failure due to image pull error",                                               │
│    "root_cause": "The image 'myapp:v1.2.3' could not be pulled because access was denied, suggesting the        │
│  repository might not exist or requires authentication.",                                                       │
│    "errors": [                                                                                                  │
│      {                                                                                                          │
│        "message": "Pod myapp-deployment-7b8c9d5f4-abc12 failed to start",                                       │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 14:32:17.890"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "Failed to pull image \"myapp:v1.2.3\": pull access denied, repository does not exist or may  │
│  require 'docker login'",                                                                                       │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 14:32:18.123"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff",                              │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 14:32:18.456"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "Deployment rollout failed: deployment \"myapp-deployment\" exceeded its progress deadline",  │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 14:32:25.901"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "Production deployment failed - rollback initiated",                                          │
│        "severity": "CRITICAL",                                                                                  │
│        "timestamp": "2024-01-15 14:32:29.456"          

In [7]:
report = structured_result.pydantic

print(f"Primary issue: {report.primary_issue}")
print(f"Root cause: {report.root_cause}")
print(f"\nErrors found: {len(report.errors)}")
for error in report.errors:
    print(f"  [{error.severity}] {error.timestamp} — {error.message}")
print(f"\nAffected components: {report.affected_components}")

Primary issue: Deployment failure due to image pull error
Root cause: The image 'myapp:v1.2.3' could not be pulled because access was denied, suggesting the repository might not exist or requires authentication.

Errors found: 5
  [ERROR] 2024-01-15 14:32:17.890 — Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
  [ERROR] 2024-01-15 14:32:18.123 — Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
  [ERROR] 2024-01-15 14:32:18.456 — Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
  [ERROR] 2024-01-15 14:32:25.901 — Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
  [CRITICAL] 2024-01-15 14:32:29.456 — Production deployment failed - rollback initiated

Affected components: ['myapp-deployment', 'myapp-deployment-7b8c9d5f4-abc12', 'myapp-service']


In [8]:
print(report.model_dump_json(indent=2))

{
  "primary_issue": "Deployment failure due to image pull error",
  "root_cause": "The image 'myapp:v1.2.3' could not be pulled because access was denied, suggesting the repository might not exist or requires authentication.",
  "errors": [
    {
      "message": "Pod myapp-deployment-7b8c9d5f4-abc12 failed to start",
      "severity": "ERROR",
      "timestamp": "2024-01-15 14:32:17.890"
    },
    {
      "message": "Failed to pull image \"myapp:v1.2.3\": pull access denied, repository does not exist or may require 'docker login'",
      "severity": "ERROR",
      "timestamp": "2024-01-15 14:32:18.123"
    },
    {
      "message": "Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff",
      "severity": "ERROR",
      "timestamp": "2024-01-15 14:32:18.456"
    },
    {
      "message": "Deployment rollout failed: deployment \"myapp-deployment\" exceeded its progress deadline",
      "severity": "ERROR",
      "timestamp": "2024-01-15 14:32:25.901"
    },
    {
      "messa

Same agent. Same log. One parameter changed everything — `output_pydantic=LogAnalysisReport`. Now every field is typed, accessible, and guaranteed to be there.

---

## 2. Code Guardrail

### The Problem: Bad Output Passes Through

Structured output guarantees the *shape*, but not the *quality*. What if the agent produces a report with zero errors identified, or a vague root cause? With no guardrail, that bad output flows straight to the next agent.

Let's use a trickier log — shorter, more ambiguous — where the agent is more likely to produce a vague analysis.

In [9]:
TRICKY_LOG_INPUT = """
[2024-01-15 09:01:22.100] INFO: Application startup sequence initiated
[2024-01-15 09:01:23.200] WARNING: Connection pool size reaching 80% capacity
[2024-01-15 09:01:24.300] WARNING: Slow query detected (2340ms) on /api/users
[2024-01-15 09:01:25.400] ERROR: Connection timeout after 30000ms
[2024-01-15 09:01:26.500] INFO: Retrying connection (attempt 2/3)
[2024-01-15 09:01:27.600] INFO: Retrying connection (attempt 3/3)
[2024-01-15 09:01:28.700] ERROR: All retry attempts exhausted
"""

unguarded_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    agent=log_analyzer,
)

unguarded_crew = Crew(
    agents=[log_analyzer],
    tasks=[unguarded_task],
    process=Process.sequential,
    verbose=False,
)

unguarded_result = unguarded_crew.kickoff(inputs={"log_data": TRICKY_LOG_INPUT})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 09:01:22.100] INFO: Application startup sequence initiated                                         │
│  [2024-01-15 09:01:23.200] WARNING: Connection pool size reaching 80% capacity                                  │
│  [2024-01-15 09:01:24.300] WARNING: Slow query detected (2340ms) on /api/users                                  │
│  [2024-01-15 09:01:25.400] ERROR: Connection timeout after 30000ms                                              │
│  [2024-01-15 09:01:26.500] INFO: Retrying connection (attempt 2/3)                                              │
│  [2024-01-15 09:01:27.600] INFO: Retrying connection (attempt 3/3)                                              │
│  [2024-01-15 09:01:28.700] ERROR: All retry attempts exhausted                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "primary_issue": "Connection timeout leading to failed retries",                                             │
│    "root_cause": "Network instability and insufficient connection pool resulting in connection timeouts.",      │
│    "errors": [                                                                                                  │
│      {                                                                                                          │
│        "message": "Connection timeout after 30000ms",                                                           │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:25.400"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "All retry attempts exhausted",                                                               │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:28.700"                                                                   │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "affected_components": [                                                                                     │
│      "Database Connection",                                                                                     │
│      "API Service"                                                                                              │
│    ],                                                                                                           │
│    "timeline": [                                                                                                │
│      "2024-01-15 09:01:22.100 - Application startup sequence initiated",                                        │
│      "2024-01-15 09:01:23.200 - Connection pool size reaching 80% capacity",                                    │
│      "2024-01-15 09:01:24.300 - Slow query detected (2340ms) on /api/users",                                    │
│      "2024-01-15 09:01:25.400 - Connection timeout after 30000ms",                                              │
│      "2024-01-15 09:01:26.500 - Retrying connection (attempt 2/3)",                                             │
│      "2024-01-15 09:01:27.600 - Retrying connection (attempt 3/3)",                                             │
│      "2024-01-15 09:01:28.700 - All retry attempts exhausted"                                                   │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                        

In [10]:
report = unguarded_result.pydantic
print(f"Errors found: {len(report.errors)}")
print(f"Root cause: {report.root_cause}")
print(f"Affected components: {report.affected_components}")

# Whatever the agent returned — good or bad — it passed through unchecked.
# In a real pipeline, the next agent would receive this as-is.

Errors found: 2
Root cause: Network instability and insufficient connection pool resulting in connection timeouts.
Affected components: ['Database Connection', 'API Service']


### The Fix: Add a Guardrail

A guardrail is a function that validates the output *before* it's accepted. It returns:
- `(True, data)` — output is good, pass it through
- `(False, "reason")` — output is rejected, agent retries automatically

Run this with `verbose=True` to see the guardrail in action.

In [11]:
def validate_log_analysis(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if not report or not report.errors:
        return (False, "Analysis must identify at least one error from the logs")
    if not report.root_cause:
        return (False, "Root cause analysis is required")
    return (True, report)

In [12]:
guarded_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    guardrail=validate_log_analysis,
    agent=log_analyzer,
)

guarded_crew = Crew(
    agents=[log_analyzer],
    tasks=[guarded_task],
    process=Process.sequential,
    verbose=True,
)

guarded_result = guarded_crew.kickoff(inputs={"log_data": TRICKY_LOG_INPUT})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d611cd21-c421-481e-912c-91b51cedb56b                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 09:01:22.100] INFO: Application startup sequence initiated                                         │
│  [2024-01-15 09:01:23.200] WARNING: Connection pool size reaching 80% capacity                                  │
│  [2024-01-15 09:01:24.300] WARNING: Slow query detected (2340ms) on /api/users                                  │
│  [2024-01-15 09:01:25.400] ERROR: Connection timeout after 30000ms                                              │
│  [2024-01-15 09:01:26.500] INFO: Retrying connection (attempt 2/3)                                              │
│  [2024-01-15 09:01:27.600] INFO: Retrying connection (attempt 3/3)                                              │
│  [2024-01-15 09:01:28.700] ERROR: All retry attempts exhausted                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "primary_issue": "Connection timeout and failed retries",                                                    │
│    "root_cause": "Excessive connection pool usage leading to timeouts",                                         │
│    "errors": [                                                                                                  │
│      {                                                                                                          │
│        "message": "Connection timeout after 30000ms",                                                           │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:25.400"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "All retry attempts exhausted",                                                               │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:28.700"                                                                   │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "affected_components": [                                                                                     │
│      "Database connection"                                                                                      │
│    ],                                                                                                           │
│    "timeline": [                                                                                                │
│      "2024-01-15 09:01:22.100 - Application startup sequence initiated",                                        │
│      "2024-01-15 09:01:23.200 - Connection pool size reaching 80% capacity",                                    │
│      "2024-01-15 09:01:24.300 - Slow query detected (2340ms) on /api/users",                                    │
│      "2024-01-15 09:01:25.400 - Connection timeout after 30000ms",                                              │
│      "2024-01-15 09:01:26.500 - Retrying connection (attempt 2/3)",                                             │
│      "2024-01-15 09:01:27.600 - Retrying connection (attempt 3/3)",                                             │
│      "2024-01-15 09:01:28.700 - All retry attempts exhausted"                                                   │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def validate_log_analysis(result: TaskOutput) -> T...                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 5673cd5e-b174-40c3-9d13-03aef9a5751b                                                                     │
│  Agent: DevOps Log Analyzer                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d611cd21-c421-481e-912c-91b51cedb56b                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "primary_issue": "Connection timeout and failed retries",                                                    │
│    "root_cause": "Excessive connection pool usage leading to timeouts",                                         │
│    "errors": [                                                                                                  │
│      {                                                                                                          │
│        "message": "Connection timeout after 30000ms",                                                           │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:25.400"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "All retry attempts exhausted",                                                               │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:28.700"                                                                   │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "affected_components": [                                                                                     │
│      "Database connection"                                                                                      │
│    ],                                                                                                           │
│    "timeline": [                                                                                                │
│      "2024-01-15 09:01:22.100 - Application startup sequence initiated",                                        │
│      "2024-01-15 09:01:23.200 - Connection pool size reaching 80% capacity",                                    │
│      "2024-01-15 09:01:24.300 - Slow query detected (2340ms) on /api/users",                                    │
│      "2024-01-15 09:01:25.400 - Connection timeout after 30000ms",                                              │
│      "2024-01-15 09:01:26.500 - Retrying connection (attempt 2/3)",                                             │
│      "2024-01-15 09:01:27.600 - Retrying connection (attempt 3/3)",                                             │
│      "2024-01-15 09:01:28.700 - All retry attempts exhausted"                                                   │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                       

In [13]:
report = guarded_result.pydantic
print(f"Errors found: {len(report.errors)}")
print(f"Root cause: {report.root_cause}")
for error in report.errors:
    print(f"  [{error.severity}] {error.timestamp} — {error.message}")

Errors found: 2
Root cause: Excessive connection pool usage leading to timeouts
  [ERROR] 2024-01-15 09:01:25.400 — Connection timeout after 30000ms
  [ERROR] 2024-01-15 09:01:28.700 — All retry attempts exhausted


---

## 3. No-Code Guardrail

Writing validation functions for everything is tedious. For simpler checks, you can pass a **plain English string** as the guardrail. CrewAI uses an LLM to evaluate whether the output meets your criteria.

In [15]:
solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure issues.""",
    verbose=True,
)

solution_task = Task(
    description="Provide a solution for the following issue: {issue}",
    expected_output="A remediation plan with specific commands",
    guardrail="The solution must include at least 3 specific, copy-pasteable shell commands. "
    "Reject if it only contains general advice without concrete commands.",
    agent=solution_specialist,
)

solution_crew = Crew(
    agents=[solution_specialist],
    tasks=[solution_task],
    process=Process.sequential,
    verbose=True,
)

solution_result = solution_crew.kickoff(
    inputs={"issue": "Kubernetes pods failing with ImagePullBackOff due to missing registry credentials"}
)
print(f"\nFinal solution:\n{solution_result.raw}")

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cedb4300-8fdc-46b6-8e40-63b41eecbbbe                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Solution Specialist                                                                              │
│                                                                                                                 │
│  Task: Provide a solution for the following issue: Kubernetes pods failing with ImagePullBackOff due to         │
│  missing registry credentials                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Solution Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To remediate the issue of Kubernetes pods failing with ImagePullBackOff due to missing registry credentials,   │
│  follow this step-by-step remediation plan:                                                                     │
│                                                                                                                 │
│  1. **Identify the Image and Registry**:                                                                        │
│     - Check which image the pod is trying to pull by reviewing the pod specifications in your Kubernetes        │
│  deployment or pod YAML file. Take note of the image name and the associated registry.                          │
│                                                                                                                 │
│  2. **Verify Existing Secrets**:                                                                                │
│     - Check if a Kubernetes Secret containing registry credentials exists in the same namespace where the pods  │
│  are deployed. This can often be named something like `regcred` or similar.                                     │
│                                                                                                                 │
│  3. **Create or Update Docker Registry Secret**:                                                                │
│     - If no secret exists, or if existing credentials are outdated, create or update a Kubernetes Secret using  │
│  valid credentials for the registry. This Secret should be of type `docker-registry`.                           │
│                                                                                                                 │
│  4. **Define the Secret in Pod or ServiceAccount**:                                                             │
│     - Once the Secret is created or updated, ensure that the Kubernetes deployment, pod, or the associated      │
│  `ServiceAccount` has the necessary information to use this Secret.                                             │
│     - In the pod or deployment specification, under `spec.template.spec.imagePullSecrets`, make sure the name   │
│  of the Secret is referenced correctly.                                                                         │
│                                                                                                                 │
│  5. **Validate Access and Authentication**:                                                                     │
│     - Manually attempt to access the container registry using the same credentials outside of Kubernetes to     │
│  ensure that the issue is not with the credentials themselves.                                                  │
│                                                                                                                 │
│  6. **Deploy the Configuration Changes**:                                                                       │
│     - Update the Kubernetes resources with these changes. This might involve redeploying the deployment or the  │
│  pod to ensure the configuration changes take effect.                                                           │
│                                                                                                                 │
│  7. **Monitor Pod Status**:                            

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: The solution must include at least 3 specific, cop...                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 🛡️ Guardrail Failed ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Failed                                                                                               │
│  Name: Validation Error                                                                                         │
│  Error: The task result does not comply with the guardrail as it lacks at least 3 specific, copy-pasteable      │
│  shell commands. The result contains a step-by-step plan, but does not include concrete shell commands for      │
│  creating or updating registry secrets, referencing the secret in deployment, or any specific Kubernetes        │
│  commands for diagnostics or redeployment. To comply, the solution should have included specific shell          │
│  commands for these actions.                                                                                    │
│  Attempts: 3                                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Guardrail blocked, retrying, due to: The task result does not comply with the guardrail as it lacks at least 3 specific, copy-pasteable shell commands. The result contains a step-by-step plan, but does not include concrete shell commands for creating or updating registry secrets, referencing the secret in deployment, or any specific Kubernetes commands for diagnostics or redeployment. To comply, the solution should have included specific shell commands for these actions.



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Solution Specialist                                                                              │
│                                                                                                                 │
│  Task: Provide a solution for the following issue: Kubernetes pods failing with ImagePullBackOff due to         │
│  missing registry credentials                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

KeyboardInterrupt: 

---

## 4. Memory

### The Problem: Every Run Starts From Scratch

Run the log analyzer on two different logs back-to-back. The second analysis has zero reference to the first — the agent starts completely fresh every time.

In [16]:
DB_LOG_INPUT = """
[2024-01-16 03:15:10.100] INFO: Database backup job started
[2024-01-16 03:15:11.200] WARNING: Replication lag detected (15s behind primary)
[2024-01-16 03:15:12.300] ERROR: Connection refused to replica db-replica-02:5432
[2024-01-16 03:15:13.400] ERROR: Authentication failed for user 'backup_agent' on db-primary:5432
[2024-01-16 03:15:14.500] CRITICAL: Backup job failed - unable to connect to any database instance
"""

memory_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

no_memory_crew = Crew(
    agents=[log_analyzer],
    tasks=[memory_task],
    process=Process.sequential,
    verbose=False,
)

In [17]:
print("=" * 60)
print("RUN 1: Kubernetes deployment log")
print("=" * 60)
r1 = no_memory_crew.kickoff(inputs={"log_data": LOG_INPUT})
print(r1.raw)

============================================================

RUN 1: Kubernetes deployment log

============================================================

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Detailed Analysis Report**                                                                                   │
│                                                                                                                 │
│  **Primary Issue Description:**                                                                                 │
│  The deployment of `myapp-deployment` failed due to an inability to pull the required Docker image, leading to  │
│  several cascading failures in the deployment process.                                                          │
│                                                                                                                 │
│  **Key Error Messages:**                                                                                        │
│  1. `[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start`                     │
│  2. `[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does  │
│  not exist or may require 'docker login'`                                                                       │
│  3. `[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`            │
│  4. `[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its     │
│  progress deadline`                                                                                             │
│  5. `[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated`                     │
│                                                                                                                 │
│  **Root Cause Analysis:**                                                                                       │
│  The deployment failed primarily due to an issue with pulling the Docker image `myapp:v1.2.3`. The error        │
│  indicates a possible lack of access rights needed to pull the image, suggesting that the image repository may  │
│  not be accessible without proper authentication, such as a `docker login`. This resulted in the pod            │
│  `myapp-deployment-7b8c9d5f4-abc12` not being able to start, entering a `Pending` state initially and then      │
│  falling into an `ImagePullBackOff` state, indicating retries for pulling the image, which were unsuccessful.   │
│  Consequently, the deployment exceeded its progress deadline as it was unable to correctly initialize due to    │
│  the image pull failure.                                                                                        │
│                                                                                                                 │
│  **Affected Components:**                                                                                       │
│  1. **Pods:** Specifically, the pod `myapp-deployment-7b8c9d5f4-abc12` is directly affected. It failed to       │
│  start due to the image pull error.                                                                             │
│  2. **Deployment Manager:** The deployment `myapp-deployment` exceeded its progress deadline, meaning the       │
│  service was unable to reach a consistent operational state and complete rollout as planned.                    │
│  3. **Application Service:** The service `myapp-service

**Detailed Analysis Report**

**Primary Issue Description:**  
The deployment of `myapp-deployment` failed due to an inability to pull the required Docker image, leading to several cascading failures in the deployment process.

**Key Error Messages:**  
1. `[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start`
2. `[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'`
3. `[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`
4. `[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline`
5. `[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated`

**Root Cause Analysis:**  
The deployment failed primarily due to an issue with pulling the Docker image `myapp:v1.2.3`. The error indicates a possible lack of access rights needed to pu

In [18]:
print("=" * 60)
print("RUN 2: Database backup log (no memory of Run 1)")
print("=" * 60)
r2 = no_memory_crew.kickoff(inputs={"log_data": DB_LOG_INPUT})
print(r2.raw)

# The second analysis has zero reference to the Kubernetes issue.
# The agent starts from scratch every time.

RUN 2: Database backup log (no memory of Run 1)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-16 03:15:10.100] INFO: Database backup job started                                                    │
│  [2024-01-16 03:15:11.200] WARNING: Replication lag detected (15s behind primary)                               │
│  [2024-01-16 03:15:12.300] ERROR: Connection refused to replica db-replica-02:5432                              │
│  [2024-01-16 03:15:13.400] ERROR: Authentication failed for user 'backup_agent' on db-primary:5432              │
│  [2024-01-16 03:15:14.500] CRITICAL: Backup job failed - unable to connect to any database instance             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Primary Issue Description:                                                                                     │
│  The primary issue is a failure of the database backup job due to connectivity and authentication issues with   │
│  the database instances.                                                                                        │
│                                                                                                                 │
│  Key Error Messages:                                                                                            │
│  1. `[2024-01-16 03:15:12.300] ERROR: Connection refused to replica db-replica-02:5432`                         │
│  2. `[2024-01-16 03:15:13.400] ERROR: Authentication failed for user 'backup_agent' on db-primary:5432`         │
│  3. `[2024-01-16 03:15:14.500] CRITICAL: Backup job failed - unable to connect to any database instance`        │
│                                                                                                                 │
│  Root Cause Analysis:                                                                                           │
│  The log data reveals two critical issues that have led to the failure of the database backup job:              │
│  - The backup job is unable to establish a connection to the replica database `db-replica-02`. The "Connection  │
│  refused" error indicates there may be network-related issues, improper configuration, or that the database     │
│  service on the replica might not be running.                                                                   │
│  - Additionally, there is an authentication problem when trying to connect to the primary database              │
│  `db-primary`. The "Authentication failed for user 'backup_agent'" error suggests incorrect credentials or      │
│  privilege issues for the user 'backup_agent'.                                                                  │
│                                                                                                                 │
│  These two issues combined have caused the backup job to completely fail, as neither the primary nor the        │
│  replica databases were accessible for the backup operation.                                                    │
│                                                                                                                 │
│  Affected Components:                                                                                           │
│  1. **db-replica-02** – A network or service issue that is causing connection refusal.                          │
│  2. **db-primary** – Authentication issue with the user 'backup_agent', preventing access for backup purposes.  │
│  3. **Backup Job Service** – Fails to complete its operation because it cannot connect to the databases due to  │
│  the above errors.                                                                                              │
│                                                                                                                 │
│  Recommended Actions:                                                                                           │
│  - Investigate and resolve the network connectivity issues or service status for `db-replica-02`.               │
│  - Verify and correct the credentials/configuration for

Primary Issue Description:
The primary issue is a failure of the database backup job due to connectivity and authentication issues with the database instances.

Key Error Messages:
1. `[2024-01-16 03:15:12.300] ERROR: Connection refused to replica db-replica-02:5432`
2. `[2024-01-16 03:15:13.400] ERROR: Authentication failed for user 'backup_agent' on db-primary:5432`
3. `[2024-01-16 03:15:14.500] CRITICAL: Backup job failed - unable to connect to any database instance`

Root Cause Analysis:
The log data reveals two critical issues that have led to the failure of the database backup job:
- The backup job is unable to establish a connection to the replica database `db-replica-02`. The "Connection refused" error indicates there may be network-related issues, improper configuration, or that the database service on the replica might not be running.
- Additionally, there is an authentication problem when trying to connect to the primary database `db-primary`. The "Authentication failed for 

### The Fix: `memory=True`

One parameter on the Crew. The agent now remembers context from previous runs and can reference patterns it saw before.

In [19]:
memory_crew = Crew(
    agents=[log_analyzer],
    tasks=[memory_task],
    process=Process.sequential,
    memory=True,
    verbose=False,
)

In [20]:
print("=" * 60)
print("RUN 1: Kubernetes deployment log (building memory)")
print("=" * 60)
r1 = memory_crew.kickoff(inputs={"log_data": LOG_INPUT})
print(r1.raw)

ERROR:root:Error during short_term search: APIRemovedInV1.__init__() takes 1 positional argument but 2 were given
ERROR:root:Error during entities search: APIRemovedInV1.__init__() takes 1 positional argument but 2 were given


RUN 1: Kubernetes deployment log (building memory)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Detailed Analysis Report:**                                                                                  │
│                                                                                                                 │
│  1. **Primary Issue Description:**                                                                              │
│     The deployment of `myapp-deployment` failed due to issues related to image pulling. The pod failed to       │
│  start as the image required for the deployment could not be pulled from the repository, resulting in a status  │
│  of `ImagePullBackOff`. This ultimately led to the deployment exceeding its progress deadline.                  │
│                                                                                                                 │
│  2. **Key Error Messages:**                                                                                     │
│     - `[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start`                   │
│     - `[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository     │
│  does not exist or may require 'docker login'`                                                                  │
│     - `[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`          │
│     - `[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its   │
│  progress deadline`                                                                                             │
│     - `[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated`                   │
│                                                                                                                 │
│  3. **Root Cause Analysis:**                                                                                    │
│     The root cause of the deployment failure appears to relate to the inability to pull the necessary Docker    │
│  image `myapp:v1.2.3`. The error message indicates a pull access denied error, suggesting the image might not   │
│  exist in the repository or the deployment system lacks the necessary authentication credentials (could         │
│  require `docker login`). Without successfully retrieving the image, the deployment cannot proceed, causing     │
│  the pod status to change to `ImagePullBackOff`. Consequently, the deployment could not complete within the     │
│  allocated time, triggering a rollback.                                                                         │
│                                                                                                                 │
│  4. **Affected Components:**                                                                                    │
│     - **Pod:** `myapp-deployment-7b8c9d5f4-abc12`                                                               │
│     - **Image Repository:** The Docker image repository housing `myapp:v1.2.3`                                  │
│     - **Deployment:** The entire `myapp-deployment`                                                             │
│     - **Service:** `myapp-service`, which has no available endpoints due to the failed deployment               │
│                                                        

ERROR:root:Error during short_term save: APIRemovedInV1.__init__() takes 1 positional argument but 2 were given







































**Detailed Analysis Report:**

1. **Primary Issue Description:**
   The deployment of `myapp-deployment` failed due to issues related to image pulling. The pod failed to start as the image required for the deployment could not be pulled from the repository, resulting in a status of `ImagePullBackOff`. This ultimately led to the deployment exceeding its progress deadline.

2. **Key Error Messages:**
   - `[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start`
   - `[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'`
   - `[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`
   - `[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline`
   - `[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback ini

In [ ]:
print("=" * 60)
print("RUN 2: Database backup log (agent remembers Run 1)")
print("=" * 60)
r2 = memory_crew.kickoff(inputs={"log_data": DB_LOG_INPUT})
print(r2.raw)

# Compare this with the no-memory Run 2 above.
# The agent may now reference patterns from the Kubernetes analysis,
# draw comparisons, or give richer infrastructure-wide insights.

---

## Recap

| Feature | Without (Notebook 1) | With (This Notebook) | One-Line Change |
|---------|---------------------|---------------------|----------------|
| **Structured Output** | Raw markdown text, fragile parsing | Typed Python objects, guaranteed schema | `output_pydantic=Model` |
| **Code Guardrail** | Bad output passes through silently | Agent retries until output is valid | `guardrail=validate_fn` |
| **No-Code Guardrail** | Write validation code for every check | Describe the check in plain English | `guardrail="must have 3 commands"` |
| **Memory** | Every run starts from scratch | Agent learns across runs | `memory=True` |

Check out `v2/outcomes.md` for 10 real-world project ideas using these features.